# 02 — Model Training
**Explainable Multimodal Quantum-GNN Framework for Multi-Disease Chest X-ray Diagnosis**

This notebook:
1. Sets up the environment and checks GPU
2. Mounts Drive / clones the repo
3. Loads splits (from `data/splits/`) and the HuggingFace dataset
4. Runs all 8 experiment variants in sequence
5. Compares results in a summary table
6. Plots training curves
7. Saves all results


## Cell 1 — Environment Setup

In [ ]:
# ── GPU check ────────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Install dependencies ─────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    'timm',
    'datasets',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'PyYAML',
    'Pillow',
    'pennylane',
    'pennylane-lightning',
    'torch-geometric',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed.')

## Cell 2 — Mount Drive / Clone Repo

In [ ]:
import os

# ── Option A: Google Drive (if repo is stored there) ─────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/urop'   # adjust to your Drive path
    if os.path.exists(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Using repo from Drive: {REPO_PATH}')
    else:
        raise FileNotFoundError(f'Repo not found at {REPO_PATH}')
except Exception as e:
    print(f'Drive mount skipped ({e}). Trying git clone...')

    # ── Option B: Clone from GitHub ─────────────────────────────────────────
    REPO_URL  = 'https://github.com/pchandraswaroop/UROP.git'
    REPO_PATH = '/content/UROP'
    if not os.path.exists(REPO_PATH):
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_PATH])
    os.chdir(REPO_PATH)
    print(f'Repo at: {REPO_PATH}')

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print('Working directory:', os.getcwd())

## Cell 3 — Load Splits and HuggingFace Dataset

In [ ]:
import pandas as pd
from pathlib import Path
import yaml

# ── Config ───────────────────────────────────────────────────────────────────
with open('config/config.yaml') as f:
    CFG = yaml.safe_load(f)

SPLIT_DIR = CFG['splits']['split_dir']   # 'data/splits'
HF_REPO   = CFG['dataset']['hf_repo']   # 'chehablab/NIHChestXR'
HF_CACHE  = CFG['dataset']['cache_dir'] # 'data/raw'

# ── Load splits ───────────────────────────────────────────────────────────────
train_df = pd.read_csv(f'{SPLIT_DIR}/train.csv')
val_df   = pd.read_csv(f'{SPLIT_DIR}/val.csv')
test_df  = pd.read_csv(f'{SPLIT_DIR}/test.csv')
print(f'Splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')

# ── Load HuggingFace dataset ──────────────────────────────────────────────────
from datasets import load_dataset
print('Loading HuggingFace dataset (first run will download ~40 GB) ...')
hf_dataset = load_dataset(HF_REPO, split='train', cache_dir=HF_CACHE)
print(f'HF dataset: {len(hf_dataset):,} rows')

## Cell 4 — Run All Experiments

In [ ]:
import logging, json
from pathlib import Path
from src.training.experiment import Experiment

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S')

# ── Shared experiment config ──────────────────────────────────────────────────
BASE_CFG = {
    'batch_size':       32 if device == 'cuda' else 4,
    'num_workers':       4 if device == 'cuda' else 0,
    'learning_rate':    1e-4,
    'weight_decay':     1e-5,
    'amp':              device == 'cuda',
    'pretrained':       True,
    'quantum_enabled':  device == 'cuda',   # only use quantum on GPU
    'quantum_output_dim': 64,
    'clinical_output_dim': 64,
    'fused_dim':        128,
    'num_classes':      14,
    'use_weighted_loss': True,
    'pos_weight_cap':   10.0,
    'early_stopping_patience': 10,
    'save_top_k':       3,
    'checkpoint_dir':   'checkpoints',
    'log_dir':          'logs',
    'image_size':       224,
}
MAX_EPOCHS = 50

EXPERIMENTS = [
    'cnn_only',
    'vit_only',
    'quantum_cnn',
    'mlp_clinical_only',
    'gnn_only',
    'classical_multimodal',
    'quantum_gnn_multimodal',
    'shared_fusion',
]

all_results = {}   # name → {val_metrics, test_metrics}

for exp_name in EXPERIMENTS:
    print(f'\n{'='*60}')
    print(f'  Experiment: {exp_name}')
    print(f'{'='*60}')

    exp = Experiment(name=exp_name, cfg=BASE_CFG.copy(), device=device)
    exp.setup(train_df, val_df, test_df, hf_dataset)

    val_m  = exp.run(max_epochs=MAX_EPOCHS)
    test_m = exp.evaluate_test()
    exp.save_results('results')

    all_results[exp_name] = {
        'val':  val_m,
        'test': test_m,
    }
    print(f'  Val  AUC={val_m.get("auc_mean", float("nan")):.4f}  F1={val_m.get("f1_macro", float("nan")):.4f}')
    print(f'  Test AUC={test_m.get("auc_mean", float("nan")):.4f}  F1={test_m.get("f1_macro", float("nan")):.4f}')

print('\nAll experiments complete!')

## Cell 5 — Compare Results Table

In [ ]:
import pandas as pd
import numpy as np

rows = []
for exp_name, result in all_results.items():
    vm = result['val']
    tm = result['test']
    rows.append({
        'Experiment':   exp_name,
        'Val AUC':      round(vm.get('auc_mean',  float('nan')), 4),
        'Val AP':       round(vm.get('ap_mean',   float('nan')), 4),
        'Val F1':       round(vm.get('f1_macro',  float('nan')), 4),
        'Test AUC':     round(tm.get('auc_mean',  float('nan')), 4),
        'Test AP':      round(tm.get('ap_mean',   float('nan')), 4),
        'Test F1':      round(tm.get('f1_macro',  float('nan')), 4),
    })

results_df = pd.DataFrame(rows).set_index('Experiment')
print(results_df.to_string())

# Save summary CSV
results_df.to_csv('results/summary_table.csv')
print('\nSaved: results/summary_table.csv')

results_df

## Cell 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

ncols = 3
nrows = (len(EXPERIMENTS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, exp_name in zip(axes, EXPERIMENTS):
    csv_path = Path(f'logs/{exp_name}/metrics.csv')
    if not csv_path.exists():
        ax.set_title(f'{exp_name}\n(no metrics.csv)')
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue

    df = pd.read_csv(csv_path)
    ax.plot(df['epoch'], df['train_loss'], label='Train Loss', color='steelblue')
    ax.plot(df['epoch'], df['val_loss'],   label='Val Loss',   color='tomato')
    ax2 = ax.twinx()
    ax2.plot(df['epoch'], df['val_auc_mean'], label='Val AUC',
             color='forestgreen', linestyle='--')
    ax.set_title(exp_name, fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax2.set_ylabel('AUC')
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)

plt.suptitle('Training Curves — All Experiments', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: results/training_curves.png')

## Cell 7 — Save All Results

In [ ]:
import json
from pathlib import Path
from datetime import datetime

Path('results').mkdir(exist_ok=True)

# Full results JSON
results_path = 'results/all_results.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'Saved: {results_path}')

# Timestamp run
ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
meta = {
    'timestamp': ts,
    'device':    device,
    'experiments': EXPERIMENTS,
}
with open('results/run_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('All results saved to results/')
print(f'Run completed at {ts}')